<a href="https://colab.research.google.com/github/radhakalra18/AI_DATA_ANALYST_AGENT/blob/main/AI_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **LLM-POWERED AI DATA ANYALYST AGENT WITH TOOL CALLING**

In [2]:
# ================================================================
# AI DATA ANALYST AGENT
# GROQ LLM + FUNCTION CALLING + DATA ANALYSIS + PLOTLY
# ================================================================
# 1. INSTALL LIBRARIES
!pip install -q groq pandas numpy plotly openpyxl

import os
import json
import inspect
import pandas as pd
import numpy as np
import plotly.express as px
from groq import Groq
from google.colab import files
from google.colab import userdata
print("=" * 60)
print("       AI DATA ANALYST AGENT")
print("=" * 60)
# ================================================================
# 2. UPLOAD DATASET
# ================================================================

print("\nUpload your CSV or Excel dataset:")

uploaded = files.upload()

if not uploaded:
    raise ValueError("No dataset uploaded.")

filename = next(iter(uploaded))

if filename.lower().endswith(".csv"):
    try:
        df = pd.read_csv(filename, encoding="utf-8")
    except UnicodeDecodeError:
        print("UTF-8 decoding failed. Retrying with 'latin-1' encoding...")
        df = pd.read_csv(filename, encoding="latin-1")
elif filename.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(filename)
else:
    raise ValueError("Please upload a CSV or Excel file.")

print("\nDataset loaded successfully!")
print("File:", filename)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:", df.columns.tolist())


# ================================================================
# 3. GROQ API CONNECTION
# ================================================================
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if GROQ_API_KEY:
    print("API Key loaded successfully!")
else:
    raise ValueError("Please add GROQ_API_KEY in Colab Secrets.")

client = Groq(api_key=GROQ_API_KEY)

MODEL = "openai/gpt-oss-120b"

print("\nGroq connection initialized!")
print("Model:", MODEL)


# ================================================================
# 4. HELPER FUNCTIONS
# ================================================================

def normalize_name(name):
    return str(name).lower().replace(" ", "").replace("_", "").replace("-", "")


def get_column(*possible_names):

    for name in possible_names:
        for col in df.columns:
            if normalize_name(col) == normalize_name(name):
                return col

    raise ValueError(
        f"Column not found. Expected one of {possible_names}. "
        f"Available columns: {df.columns.tolist()}"
    )


def numeric_sales():

    col = get_column(
        "Sales", "Sales_INR", "Revenue",
        "Total_Sales", "Amount"
    )

    return pd.to_numeric(df[col], errors="coerce")


def json_safe(data):
    return json.loads(json.dumps(data, default=str))


# ================================================================
# 5. DATASET INSPECTION TOOL
# ================================================================

def inspect_dataset():

    return {
        "total_rows": int(df.shape[0]),
        "total_columns": int(df.shape[1]),
        "column_names": df.columns.tolist(),
        "missing_values": df.isnull().sum().to_dict(),
        "data_types": df.dtypes.astype(str).to_dict(),
        "duplicate_rows": int(df.duplicated().sum())
    }


# ================================================================
# 6. COMPLETE DATASET SUMMARY TOOL
# ================================================================

def describe_dataset():

    numeric = df.select_dtypes(include=np.number)

    return {
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "column_names": df.columns.tolist(),
        "missing_values": df.isnull().sum().to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "statistical_summary": numeric.describe().round(2).to_dict(),
        "first_five_records": df.head(5).to_dict(orient="records")
    }


# ================================================================
# 7. TOTAL SALES TOOL
# ================================================================

def calculate_total_sales():

    sales = numeric_sales()

    return {
        "total_sales": round(float(sales.sum()), 2),
        "average_sales": round(float(sales.mean()), 2),
        "minimum_sales": round(float(sales.min()), 2),
        "maximum_sales": round(float(sales.max()), 2)
    }


# ================================================================
# 8. SALES BY REGION TOOL
# ================================================================

def sales_by_region():

    region_col = get_column("Region", "Sales_Region")

    temp = pd.DataFrame({
        "Region": df[region_col],
        "Sales": numeric_sales()
    })

    result = temp.groupby("Region")["Sales"].sum()
    result = result.sort_values(ascending=False)

    return {
        str(k): round(float(v), 2)
        for k, v in result.items()
    }


# ================================================================
# 9. BAR CHART - SALES BY REGION
# ================================================================

def plot_sales_by_region():

    data = sales_by_region()

    fig = px.bar(
        x=list(data.keys()),
        y=list(data.values()),
        title="Total Sales by Region",
        labels={
            "x": "Region",
            "y": "Total Sales"
        },
        text_auto=".2s"
    )

    fig.update_layout(
        template="plotly_white",
        title_x=0.5
    )

    fig.show()

    return {
        "chart": "Regional Sales Bar Chart",
        "data": data
    }


# ================================================================
# 10. LINE CHART - MONTHLY SALES TREND
# ================================================================

def plot_sales_trend():

    date_col = get_column(
        "Order_Date", "Order Date", "OrderDate"
    )

    temp = pd.DataFrame({
        "Date": pd.to_datetime(
            df[date_col], errors="coerce"
        ),
        "Sales": numeric_sales()
    })

    temp = temp.dropna()

    if temp.empty:
        raise ValueError("No valid dates or sales values found.")

    temp["Month"] = temp["Date"].dt.to_period("M").astype(str)

    monthly = (
        temp.groupby("Month")["Sales"]
        .sum()
        .reset_index()
    )

    fig = px.line(
        monthly,
        x="Month",
        y="Sales",
        markers=True,
        title="Monthly Sales Trend"
    )

    fig.update_layout(
        template="plotly_white",
        title_x=0.5,
        xaxis_title="Month",
        yaxis_title="Total Sales"
    )

    fig.show()

    return {
        "chart": "Monthly Sales Line Chart",
        "monthly_sales": monthly.to_dict(orient="records")
    }


# ================================================================
# 11. PIE CHART - CATEGORY SALES
# ================================================================

def plot_category_sales():

    category_col = get_column(
        "Category", "Product_Category"
    )

    temp = pd.DataFrame({
        "Category": df[category_col],
        "Sales": numeric_sales()
    })

    category_sales = (
        temp.groupby("Category")["Sales"]
        .sum()
        .sort_values(ascending=False)
    )

    fig = px.pie(
        names=category_sales.index.astype(str),
        values=category_sales.values,
        title="Sales Distribution by Category",
        hole=0.3
    )

    fig.update_layout(
        template="plotly_white",
        title_x=0.5
    )

    fig.show()

    return {
        "chart": "Category Sales Pie Chart",
        "category_sales": {
            str(k): round(float(v), 2)
            for k, v in category_sales.items()
        }
    }


# ================================================================
# 12. REGISTER PYTHON FUNCTIONS
# ================================================================

available_functions = {

    "inspect_dataset": inspect_dataset,

    "describe_dataset": describe_dataset,

    "calculate_total_sales": calculate_total_sales,

    "sales_by_region": sales_by_region,

    "plot_sales_by_region": plot_sales_by_region,

    "plot_sales_trend": plot_sales_trend,

    "plot_category_sales": plot_category_sales

}


# ================================================================
# 13. REGISTER GROQ TOOLS
# ================================================================

def make_tool(name, description):

    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }


tools = [

    make_tool(
        "inspect_dataset",
        "Inspect dataset rows, columns, missing values and data types."
    ),

    make_tool(
        "describe_dataset",
        "Give a complete statistical summary of the dataset."
    ),

    make_tool(
        "calculate_total_sales",
        "Calculate total, average, minimum and maximum sales."
    ),

    make_tool(
        "sales_by_region",
        "Calculate and rank total sales by region."
    ),

    make_tool(
        "plot_sales_by_region",
        "Create a bar chart showing sales by region."
    ),

    make_tool(
        "plot_sales_trend",
        "Create a line chart showing monthly sales trends."
    ),

    make_tool(
        "plot_category_sales",
        "Create a pie chart showing sales by product category."
    )

]


# ================================================================
# 14. AI AGENT - AUTOMATIC FUNCTION CALLING
# ================================================================

def run_agent(user_query, max_iterations=5):

    messages = [

        {
            "role": "system",
            "content": """
            You are an intelligent AI Data Analyst Agent.

            Analyse the uploaded dataset using registered tools.

            Select the appropriate tool based on the user's question.

            Never invent numerical results.

            For chart requests, execute the relevant visualization tool.

            Use actual tool results to explain your findings.

            After receiving tool results, provide a clear final answer.

            Never invent tool names or raw XML tool calls.
            """
        },

        {
            "role": "user",
            "content": user_query
        }

    ]

    # Agent execution loop

    for iteration in range(max_iterations):

        print("\n" + "-" * 55)
        print("AGENT ITERATION:", iteration + 1)
        print("-" * 55)

        response = client.chat.completions.create(

            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0,
            max_tokens=500

        )

        assistant_message = response.choices[0].message

        # Check if LLM wants a tool

        if not assistant_message.tool_calls:

            print("\nTASK COMPLETED!")

            return assistant_message.content or "Analysis completed."

        # Store assistant tool-call response

        messages.append(
            assistant_message.model_dump(exclude_none=True)
        )

        # Execute selected tools

        for tool_call in assistant_message.tool_calls:

            tool_name = tool_call.function.name

            try:
                arguments = json.loads(
                    tool_call.function.arguments or "{}"
                )

                if not isinstance(arguments, dict):
                    raise ValueError("Arguments must be a dictionary.")

                # Remove invalid empty keys
                arguments = {
                    k: v for k, v in arguments.items()
                    if isinstance(k, str) and k.strip()
                }

                print("\nSELECTED TOOL:", tool_name)
                print("ARGUMENTS:", arguments)

                function = available_functions.get(tool_name)

                if function is None:

                    result = {
                        "error": "Unknown tool: " + tool_name
                    }

                else:

                    # Validate arguments
                    inspect.signature(function).bind(**arguments)

                    # Execute Python function
                    result = function(**arguments)

                print("TOOL EXECUTED SUCCESSFULLY")

            except Exception as error:

                result = {
                    "error": str(error)
                }

                print("TOOL ERROR:", error)

            # Send tool result back to LLM

            messages.append({

                "role": "tool",

                "tool_call_id": tool_call.id,

                "name": tool_name,

                "content": json.dumps(
                    json_safe(result),
                    default=str
                )

            })

    return "Maximum iterations reached. Please try a simpler question."


# ================================================================
# 15. INTERACTIVE AI ANALYST CHAT
# ================================================================

print("\n" + "=" * 60)
print("       YOUR AI DATA ANALYST AGENT IS READY!")
print("=" * 60)

print("\nAvailable capabilities:")
print("1. Dataset inspection")
print("2. Complete statistical summary")
print("3. Total sales calculation")
print("4. Regional sales analysis")
print("5. Regional bar chart")
print("6. Monthly sales trend line chart")
print("7. Category sales pie chart")

print("\nType 'exit' to stop the agent.")

while True:

    question = input("\nAsk your data question: ").strip()

    if question.lower() in ["exit", "quit", "stop"]:
        print("\nAI Data Analyst Agent stopped.")
        break

    if not question:
        print("Please enter a question.")
        continue

    try:

        answer = run_agent(question)

        print("\n" + "=" * 60)
        print("FINAL AI ANSWER")
        print("=" * 60)

        print(answer)

    except Exception as error:

        print("\nERROR:", error)
        print("Please check your question and try again.")

# ================================================================
# END OF AI DATA ANALYST AGENT
# ================================================================

       AI DATA ANALYST AGENT

Upload your CSV or Excel dataset:


Saving sample.csv to sample (1).csv
UTF-8 decoding failed. Retrying with 'latin-1' encoding...

Dataset loaded successfully!
File: sample (1).csv
Rows: 9994
Columns: 21

Column names: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
API Key loaded successfully!

Groq connection initialized!
Model: openai/gpt-oss-120b

       YOUR AI DATA ANALYST AGENT IS READY!

Available capabilities:
1. Dataset inspection
2. Complete statistical summary
3. Total sales calculation
4. Regional sales analysis
5. Regional bar chart
6. Monthly sales trend line chart
7. Category sales pie chart

Type 'exit' to stop the agent.

Ask your data question: monthly sales report

-------------------------------------------------------
AGENT ITERATION: 1
------------------------------------------------------

TOOL EXECUTED SUCCESSFULLY

-------------------------------------------------------
AGENT ITERATION: 3
-------------------------------------------------------

TASK COMPLETED!

FINAL AI ANSWER
**Monthly Sales Report**

I generated a line chart that shows total sales for each month in the dataset (January 2014 – December 2017). Below is the month‑by‑month sales figure that the chart is based on:

| Month   | Sales (USD) |
|---------|-------------|
| 2014‑01 | 14,236.90 |
| 2014‑02 | 4,519.89 |
| 2014‑03 | 55,691.01 |
| 2014‑04 | 28,295.35 |
| 2014‑05 | 23,648.29 |
| 2014‑06 | 34,595.13 |
| 2014‑07 | 33,946.39 |
| 2014‑08 | 27,909.47 |
| 2014‑09 | 81,777.35 |
| 2014‑10 | 31,453.39 |
| 2014‑11 | 78,628.72 |
| 2014‑12 | 69,545.62 |
| 2015‑01 | 18,174.08 |
| 2015‑02 | 11,951.41 |
| 2015‑03 | 38,726.25 |
| 2015‑04 | 34,195.21 |
| 2015‑05 | 30,131.69 |
| 2015‑06 | 24,797.29 |
| 2015‑07 | 28,765.33 |
| 2015‑08 | 36,898.33 |
| 2015‑09 | 64,595.92 |
| 2015‑10 | 31,404.92 |
| 2015‑11 | 75,972.56 